# 🖼️ 02 — Image Detection Pipeline

Run inference on **single images** or **batch folders** using the LIVEDET model.

This notebook reuses the backend `ObjectDetector` and `utils` modules directly — no server required.

In [ ]:
# ── Imports & Path Setup ──────────────────────────────────────────────
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
sys.path.insert(0, str(PROJECT_ROOT / "backend"))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from detector import ObjectDetector
from utils import (
    DepthEstimator,
    extract_median_depth,
    compute_real_width,
    compute_depth_cm,
    classify_severity,
    compute_heuristic_measurements,
    encode_image_base64,
)

MODEL_PATH = PROJECT_ROOT / "models" / "livedet_best.pt"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "image_detection"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Model path   : {MODEL_PATH}  (exists: {MODEL_PATH.exists()})")
print(f"Output dir   : {OUTPUT_DIR}")

In [ ]:
# ── Instantiate Detector ──────────────────────────────────────────────
detector = ObjectDetector(model_path=str(MODEL_PATH), device="cpu", conf=0.25)
print(f"✅ Detector ready — {len(detector.class_names)} classes: {detector.class_names}")

## 🔍 Single-Image Detection

Change `IMAGE_PATH` to any local image file.

In [ ]:
# ── Load & Show Original Image ────────────────────────────────────────
IMAGE_PATH = str(PROJECT_ROOT / "test_samples" / "sample.jpg")  # ← change this

img = cv2.imread(IMAGE_PATH)
if img is None:
    raise FileNotFoundError(f"Cannot read image: {IMAGE_PATH}")

print(f"Image shape: {img.shape}")
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Original Image")
plt.axis("off")
plt.show()

In [ ]:
# ── Run Detection + Annotate ──────────────────────────────────────────
result = detector.detect(img)
annotated = detector.annotate_image(img.copy(), result["detections"])

# Side-by-side comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
ax1.set_title("Original", fontsize=13)
ax1.axis("off")

ax2.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
ax2.set_title(f"Detected — {result['total_detections']} objects", fontsize=13)
ax2.axis("off")

plt.tight_layout()
plt.show()

# Print per-detection details
for i, det in enumerate(result["detections"], 1):
    m = compute_heuristic_measurements(det["bbox"], result["image_shape"])
    sev_label, sev_score = classify_severity(m["depth_cm"], m["width_cm"], det["confidence"])
    print(f"  [{i}] {det['class_name']}  conf={det['confidence']:.2f}  "
          f"depth≈{m['depth_cm']:.1f}cm  width≈{m['width_cm']:.1f}cm  "
          f"severity={sev_label} ({sev_score:.2f})")

In [ ]:
# ── Save Annotated Image ─────────────────────────────────────────────
out_name = Path(IMAGE_PATH).stem + "_detected.jpg"
out_path = OUTPUT_DIR / out_name
cv2.imwrite(str(out_path), annotated)
print(f"💾 Saved: {out_path}")

## 📂 Batch Detection

Point `BATCH_DIR` to a folder of images. All results are saved to `outputs/image_detection/`.

In [ ]:
# ── Batch Detection on a Folder ───────────────────────────────────────
import glob, time

BATCH_DIR = str(PROJECT_ROOT / "test_samples")  # ← change this
EXTENSIONS = ("*.jpg", "*.jpeg", "*.png", "*.bmp")

image_paths = []
for ext in EXTENSIONS:
    image_paths.extend(glob.glob(os.path.join(BATCH_DIR, ext)))
image_paths = sorted(image_paths)

print(f"Found {len(image_paths)} images in {BATCH_DIR}\n")

summary = []
t0 = time.time()

for img_path in image_paths:
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        print(f"  ⚠️ Skipping unreadable: {img_path}")
        continue

    res = detector.detect(img_bgr)
    ann = detector.annotate_image(img_bgr.copy(), res["detections"])

    out_name = Path(img_path).stem + "_detected.jpg"
    cv2.imwrite(str(OUTPUT_DIR / out_name), ann)

    summary.append({
        "file": os.path.basename(img_path),
        "detections": res["total_detections"],
        "classes": [d["class_name"] for d in res["detections"]],
    })
    print(f"  ✅ {os.path.basename(img_path)} → {res['total_detections']} detections")

elapsed = time.time() - t0
print(f"\n── Done: {len(summary)} images in {elapsed:.1f}s ({elapsed/max(len(summary),1):.2f}s/img)")
print(f"── Outputs saved to: {OUTPUT_DIR}")

In [ ]:
# ── Batch Results Summary Table ───────────────────────────────────────
import pandas as pd

if summary:
    df = pd.DataFrame(summary)
    df["classes"] = df["classes"].apply(lambda x: ", ".join(x) if x else "—")
    display(df.style.set_caption("Batch Detection Summary"))
else:
    print("No images processed.")